# HemaCheck - Keşifsel Veri Analizi (EDA)

**Proje:** Kan Hücresi Anomali Tespiti

**Hedef:** Kan hücrelerinin morfolojik özelliklerini kullanarak anomali tespiti yapmak.

**Veri Seti:** 5882 hücre örneği, 38 özellik, 18 farklı hücre tipi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Veri Yükleme ve Temel Bilgiler

In [ ]:
# Ana veri setini yükle
df = pd.read_csv('../data_hemacheck/blood_cell_anomaly_detection.csv')

# Referans verisini yükle
ref = pd.read_csv('../data_hemacheck/cell_type_reference.csv')

# Temel bilgiler
print("=== HEMACHECK VERİ SETİ ÖZETİ ===")
print(f"Toplam Örnek: {len(df):,}")
print(f"Özellik Sayısı: {df.shape[1]}")
print(f"\nVeri Tipleri:")
print(df.dtypes.value_counts())
print(f"\nEksik Değerler: {df.isnull().sum().sum()}")
print(f"\nDuplicate Satır: {df.duplicated().sum()}")

In [ ]:
# Kolon isimleri ve açıklamaları
print("=== KOLONLAR ===")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 2. Hedef Değişken Analizi (Anomali Tespiti)

In [ ]:
# Anomali dağılımı
anomaly_counts = df['anomaly_label'].value_counts()
print("=== ANOMALI DAĞILIMI ===")
print(f"Normal (0): {anomaly_counts[0]:,} (%{100*anomaly_counts[0]/len(df):.1f})")
print(f"Anormal (1): {anomaly_counts[1]:,} (%{100*anomaly_counts[1]/len(df):.1f})")

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(anomaly_counts.values, labels=['Normal', 'Anormal'], 
          autopct='%1.1f%%', colors=colors, startangle=90,
          textprops={'fontsize': 12, 'weight': 'bold'})
axes[0].set_title('Anomali Dağılımı', fontsize=14, weight='bold')

# Bar chart
bars = axes[1].bar(['Normal', 'Anormal'], anomaly_counts.values, color=colors, edgecolor='black')
axes[1].set_ylabel('Hücre Sayısı', fontsize=12)
axes[1].set_title('Sınıf Dağılımı', fontsize=14, weight='bold')
axes[1].set_ylim(0, max(anomaly_counts.values) * 1.1)

# Değerleri bar üzerine yaz
for bar, val in zip(bars, anomaly_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                f'{val:,}', ha='center', va='bottom', fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/01_anomaly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Hücre Tipleri ve Hastalık Kategorileri

In [ ]:
# Hücre tiplerine göre anomali dağılımı
cell_anomaly = df.groupby(['cell_type', 'anomaly_label']).size().unstack(fill_value=0)
cell_anomaly['total'] = cell_anomaly.sum(axis=1)
cell_anomaly = cell_anomaly.sort_values('total', ascending=True)

# Görselleştirme
fig, ax = plt.subplots(figsize=(12, 10))

y_pos = np.arange(len(cell_anomaly))
ax.barh(y_pos, cell_anomaly[0], color='#2ecc71', label='Normal', edgecolor='black')
ax.barh(y_pos, cell_anomaly[1], left=cell_anomaly[0], color='#e74c3c', label='Anormal', edgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels(cell_anomaly.index, fontsize=10)
ax.set_xlabel('Hücre Sayısı', fontsize=12)
ax.set_title('Hücre Tiplerine Göre Anomali Dağılımı', fontsize=14, weight='bold')
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/figures/02_cell_type_anomaly.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Hastalık kategorileri
disease_counts = df['disease_category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))

# Renkler - anomali içerenleri kırmızı tonları
colors_disease = []
for cat in disease_counts.index:
    if cat in ['Leukemia', 'Anemia', 'Infection', 'Artefact']:
        colors_disease.append('#e74c3c')
    else:
        colors_disease.append('#2ecc71')

bars = ax.bar(range(len(disease_counts)), disease_counts.values, color=colors_disease, edgecolor='black')
ax.set_xticks(range(len(disease_counts)))
ax.set_xticklabels(disease_counts.index, rotation=45, ha='right')
ax.set_ylabel('Hücre Sayısı', fontsize=12)
ax.set_title('Hastalık Kategorilerine Göre Dağılım', fontsize=14, weight='bold')

# Değerleri yaz
for i, (bar, val) in enumerate(zip(bars, disease_counts.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            f'{val}', ha='center', va='bottom', fontsize=10, weight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', edgecolor='black', label='Normal'),
                   Patch(facecolor='#e74c3c', edgecolor='black', label='Anomali')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('../reports/figures/03_disease_categories.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHastalık Kategorileri:")
for cat, count in disease_counts.items():
    pct = 100 * count / len(df)
    print(f"  {cat}: {count:,} (%{pct:.1f})")

## 4. Sayısal Özelliklerin İstatistiksel Analizi

In [ ]:
# Sayısal kolonları seç
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['anomaly_label', 'cytodiffusion_anomaly_score', 
                                                      'cytodiffusion_classification_confidence']]

# Temel istatistikler
print(f"\n=== SAYISAL ÖZELLİKLER ({len(numeric_cols)} adet) ===")
desc = df[numeric_cols].describe().T
desc['missing'] = df[numeric_cols].isnull().sum()
print(desc.round(3))

In [ ]:
# Önemli morfolojik özellikleri görselleştir
morph_features = ['cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 
                  'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score']

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(morph_features):
    ax = axes[idx]
    
    # Her iki sınıf için histogram
    normal_data = df[df['anomaly_label'] == 0][feature]
    anomaly_data = df[df['anomaly_label'] == 1][feature]
    
    ax.hist(normal_data, bins=30, alpha=0.7, color='#2ecc71', label='Normal', edgecolor='black')
    ax.hist(anomaly_data, bins=30, alpha=0.7, color='#e74c3c', label='Anormal', edgecolor='black')
    
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Frekans', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_title(f'{feature}', fontsize=11, weight='bold')

# Son subplot boş, kullanım amacı
axes[7].axis('off')
axes[7].text(0.5, 0.5, 'HemaCheck\nMorphological\nFeatures', 
            ha='center', va='center', fontsize=16, weight='bold',
            transform=axes[7].transAxes,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.suptitle('Morfolojik Özellikler: Normal vs Anormal', fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/04_morphological_features.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Korelasyon Analizi

In [ ]:
# Korelasyon matrisi - seçili özellikler
selected_features = ['cell_diameter_um', 'nucleus_area_pct', 'chromatin_density',
                    'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score',
                    'lobularity_score', 'membrane_smoothness', 'stain_intensity',
                    'anomaly_label']

corr_matrix = df[selected_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={'size': 9}, ax=ax)
ax.set_title('Özellik Korelasyon Matrisi', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Anomali ile en yüksek korelasyonlu özellikler
print("\nAnomali ile Korelasyon (En Yüksek 10):")
anomaly_corr = corr_matrix['anomaly_label'].drop('anomaly_label').abs().sort_values(ascending=False)
for feat, corr in anomaly_corr.head(10).items():
    print(f"  {feat}: {corr:.3f}")

## 6. PCA - Boyut İndirgeme ve Görselleştirme

In [ ]:
# PCA için veri hazırlığı
feature_cols = [c for c in numeric_cols if c != 'anomaly_label']
X = df[feature_cols].copy()
y = df['anomaly_label']

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Açıklanan varyans oranı
explained_var = pca.explained_variance_ratio_
cumsum_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 11), explained_var[:10] * 100, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Bileşen', fontsize=12)
axes[0].set_ylabel('Açıklanan Varyans (%)', fontsize=12)
axes[0].set_title('PCA - İlk 10 Bileşen', fontsize=13, weight='bold')
axes[0].set_xticks(range(1, 11))

# Cumulative variance
axes[1].plot(range(1, 21), cumsum_var[:20] * 100, 'o-', color='darkred', linewidth=2, markersize=6)
axes[1].axhline(y=80, color='green', linestyle='--', label='%80 Varyans')
axes[1].axhline(y=90, color='orange', linestyle='--', label='%90 Varyans')
axes[1].set_xlabel('Bileşen Sayısı', fontsize=12)
axes[1].set_ylabel('Kümülatif Varyans (%)', fontsize=12)
axes[1].set_title('Kümülatif Açıklanan Varyans', fontsize=13, weight='bold')
axes[1].legend()
axes[1].set_xticks(range(1, 21, 2))

plt.tight_layout()
plt.savefig('../reports/figures/06_pca_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nİlk 2 bileşen: %{cumsum_var[1]*100:.1f} varyans")
print(f"İlk 5 bileşen: %{cumsum_var[4]*100:.1f} varyans")
print(f"İlk 10 bileşen: %{cumsum_var[9]*100:.1f} varyans")

In [ ]:
# 2D PCA scatter plot
fig, ax = plt.subplots(figsize=(12, 8))

# Normal ve anormal için ayrı scatter
normal_mask = y == 0
anomaly_mask = y == 1

ax.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1], 
          c='#2ecc71', alpha=0.6, s=20, label='Normal', edgecolors='none')
ax.scatter(X_pca[anomaly_mask, 0], X_pca[anomaly_mask, 1], 
          c='#e74c3c', alpha=0.6, s=20, label='Anormal', edgecolors='none')

ax.set_xlabel(f'PC1 (%{explained_var[0]*100:.1f})', fontsize=12)
ax.set_ylabel(f'PC2 (%{explained_var[1]*100:.1f})', fontsize=12)
ax.set_title('PCA: Normal vs Anormal Hücreler', fontsize=14, weight='bold')
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
plt.savefig('../reports/figures/07_pca_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Hasta Demografisi Analizi

In [ ]:
# Yaş grubu ve cinsiyet analizi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Yaş grubu dağılımı
age_counts = df['patient_age_group'].value_counts()
axes[0, 0].bar(age_counts.index, age_counts.values, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Yaş Grubu Dağılımı', fontsize=12, weight='bold')
axes[0, 0].set_ylabel('Hücre Sayısı')
for i, v in enumerate(age_counts.values):
    axes[0, 0].text(i, v + 20, str(v), ha='center', fontsize=10, weight='bold')

# Cinsiyet dağılımı
sex_counts = df['patient_sex'].value_counts()
axes[0, 1].pie(sex_counts.values, labels=['Erkek' if x == 'M' else 'Kadın' for x in sex_counts.index], 
              autopct='%1.1f%%', colors=['#3498db', '#e91e63'], startangle=90)
axes[0, 1].set_title('Cinsiyet Dağılımı', fontsize=12, weight='bold')

# Yaş grubuna göre anomali oranı
age_anomaly = df.groupby('patient_age_group')['anomaly_label'].mean() * 100
bars = axes[1, 0].bar(age_anomaly.index, age_anomaly.values, color='coral', edgecolor='black')
axes[1, 0].set_title('Yaş Grubuna Göre Anomali Oranı (%)', fontsize=12, weight='bold')
axes[1, 0].set_ylabel('Anomali %')
axes[1, 0].set_ylim(0, 100)
for i, v in enumerate(age_anomaly.values):
    axes[1, 0].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, weight='bold')

# Cinsiyete göre anomali oranı
sex_anomaly = df.groupby('patient_sex')['anomaly_label'].mean() * 100
colors_sex = ['#3498db', '#e91e63']
bars = axes[1, 1].bar(['Erkek', 'Kadın'], [sex_anomaly['M'], sex_anomaly['F']], 
                      color=colors_sex, edgecolor='black')
axes[1, 1].set_title('Cinsiyete Göre Anomali Oranı (%)', fontsize=12, weight='bold')
axes[1, 1].set_ylabel('Anomali %')
axes[1, 1].set_ylim(0, 100)
for i, v in enumerate([sex_anomaly['M'], sex_anomaly['F']]):
    axes[1, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, weight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/08_demographics.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. CytoDiffusion Benchmark Karşılaştırması

In [ ]:
# Benchmark verilerini yükle
benchmark = pd.read_csv('../data_hemacheck/cytodiffusion_benchmark_scores.csv')
benchmark = benchmark.dropna(subset=['score'])
print(benchmark[['model', 'task', 'score']].to_string(index=False))

# Görselleştirme
fig, ax = plt.subplots(figsize=(12, 6))

# Model ve görev bazlı skorlar
pivot = benchmark.pivot(index='task', columns='model', values='score')
pivot.plot(kind='barh', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_xlabel('Skor', fontsize=12)
ax.set_title('CytoDiffusion Benchmark Skorları', fontsize=14, weight='bold')
ax.legend(title='Model', loc='lower right')
ax.set_xlim(0, 1.1)

plt.tight_layout()
plt.savefig('../reports/figures/09_benchmark.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. EDA Özet ve Key Findings

In [ ]:
# Özet istatistikler
print("=" * 60)
print("          HEMACHECK EDA - KEY FINDINGS")
print("=" * 60)

print(f"\n📊 VERİ SETİ:")
print(f"   • Toplam örnek: {len(df):,} hücre")
print(f"   • Normal: {anomaly_counts[0]:,} (%{100*anomaly_counts[0]/len(df):.1f})")
print(f"   • Anormal: {anomaly_counts[1]:,} (%{100*anomaly_counts[1]/len(df):.1f})")

print(f"\n🔬 HÜCRE TİPLERİ:")
print(f"   • Toplam farklı tip: {df['cell_type'].nunique()}")

print(f"\n🏥 HASTALIK KATEGORİLERİ:")
for cat in ['Normal_WBC', 'Normal_RBC', 'Leukemia', 'Anemia', 'Infection', 'Artefact']:
    count = (df['disease_category'] == cat).sum()
    print(f"   • {cat}: {count:,} hücre")

print(f"\n📈 PCA SONUÇLARI:")
print(f"   • İlk 2 bileşen: %{cumsum_var[1]*100:.1f} varyans")
print(f"   • İlk 5 bileşen: %{cumsum_var[4]*100:.1f} varyans")

print(f"\n🔗 ANOMALI İLE EN KORELE ÖZELLİKLER:")
for feat, corr in anomaly_corr.head(5).items():
    print(f"   • {feat}: {corr:.3f}")

print("\n" + "=" * 60)

## 10. Kaydedilen Dosyalar

EDA sonucu oluşturulan tüm görselleştirmeler `../reports/figures/` klasörüne kaydedildi:

1. `01_anomaly_distribution.png` - Anomali dağılımı
2. `02_cell_type_anomaly.png` - Hücre tiplerine göre dağılım
3. `03_disease_categories.png` - Hastalık kategorileri
4. `04_morphological_features.png` - Morfolojik özellikler
5. `05_correlation_matrix.png` - Korelasyon matrisi
6. `06_pca_variance.png` - PCA varyans analizi
7. `07_pca_scatter.png` - PCA scatter plot
8. `08_demographics.png` - Demografik analiz
9. `09_benchmark.png` - Benchmark karşılaştırması

---

**Sonraki Adım:** `02_feature_engineering.ipynb`